# Titanic 데이터 분류 모델 (DecisionTreeClassifier, Lv2)

seaborn `titanic` 데이터로 **생존 여부(survived)** 를 예측하는 분류 예제입니다.

- **X (독립변수)**: `pclass`, `fare`, `age`, `embarked`
- **y (종속변수)**: `survived` (0=사망, 1=생존)
- **전처리**:
  - 수치형: `SimpleImputer` + `StandardScaler`
  - 범주형: `SimpleImputer` + `OneHotEncoder`
- **모델**: `ColumnTransformer` + `DecisionTreeClassifier`


# 환경설정

In [11]:
# 필요한 라이브러리 import

import seaborn as sns
import pandas as pd
import numpy as np

# train_test_split: 데이터를 학습용 / 테스트용으로 나누기 위해 사용
from sklearn.model_selection import train_test_split

# 결측치(NaN)를 규칙에 따라 채우기 도구
from sklearn.impute import SimpleImputer

# StandardScaler : 수치형 변수의 크기(스케일)를 비슷하게 맞추기 위해 사용
# OneHotEncoder : 문자열 범주 데이터를 0/1 형태로 변형
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 수치형 컬럼, 범주형 컬럼에 서로 다른 전처리 모듈을 동시에 적용하기 위한 도구
from sklearn.compose import ColumnTransformer

# Pipeline : 전처리 + 모델 학습 과정을 하나로 묶기 위해 사용
from sklearn.pipeline import Pipeline

# 의사결정나무 모델
from sklearn.tree import DecisionTreeClassifier

# 분류 성능 평가 지표
# 이론적 근간 : 확률(조건부 확률)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# 데이터 불러오기

In [12]:
# ============================================================
# 1. 데이터 로드
# ============================================================
titanic = sns.load_dataset('titanic')

print('데이터 shape:', titanic.shape)
print()
titanic.loc[:, ['survived', 'pclass', 'fare', 'age', 'embarked']].head()


데이터 shape: (891, 15)



,survived,pclass,fare,age,embarked
0,0,3,7.2500,22.0,S
1,1,1,71.2833,38.0,C
2,1,3,7.9250,26.0,S
3,1,1,53.1000,35.0,S
4,0,3,8.0500,35.0,S


# 데이터 처리

In [13]:
# ============================================================
# 2. 전처리 / 학습·테스트 분리
# ============================================================
# 사용할 독립변수(X)
# pclass   : 객실 등급 (1등석, 2등석, 3등석)
# fare     : 승객이 지불한 요금
# age      : 승객의 나이 (결측치가 일부 존재)
# embarked : 탑승 항구 (C=Cherbourg, Q=Queenstown, S=Southampton)
X = titanic.loc[:, ['pclass', 'fare', 'age', 'embarked']]

# 종속변수(y)
# survived : 생존 여부 (0=사망, 1=생존)
y = titanic.loc[:, 'survived']

# 80% 학습, 20% 테스트 (random_state=42 로 재현 가능)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'학습 데이터: {len(X_train)}건')
print(f'테스트 데이터: {len(X_test)}건')
print()
print('결측치 개수 확인:')
print(X.isnull().sum())


학습 데이터: 712건
테스트 데이터: 179건

결측치 개수 확인:
pclass        0
fare          0
age         177
embarked      2
dtype: int64


# 데이터 학습 전 : Pipeline 정의

In [14]:
# ============================================================
# 3. ColumnTransformer + Pipeline 정의
# ============================================================
# 수치형 변수와 범주형 변수를 나눔
# pclass, fare, age 는 숫자형 데이터
# embarked 는 문자형(범주형) 데이터
numeric_features = ['pclass', 'fare', 'age'] # 결측치 존재
categorical_features = ['embarked'] # 결측치 존재

# 수치형 전처리
# 1) SimpleImputer(strategy='median')
#    -> age 같은 숫자형 결측치를 중앙값으로 채움
# 2) StandardScaler()
#    -> 변수 크기를 평균 0, 표준편차 1 기준으로 맞춤
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 범주형 전처리
# 1) SimpleImputer(strategy='most_frequent')
#    -> embarked 결측치를 가장 많이 나온 값으로 채움
# 2) OneHotEncoder(handle_unknown='ignore')
#    -> C / Q / S 같은 문자값을 0과 1로 분리된 컬럼으로 변환
#    -> 예: embarked_C, embarked_Q, embarked_S
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown="ignore"))
])


# ColumnTransformer : 패키징 도구
# -> 어떤 컬럼에는 수치형 전처리, 어떤 컬럼에는 범주형 전처리를 적용할지 지정
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# 최종 Pipeline
# 1) preprocessor에서 전처리 수행
# 2) 전처리된 데이터를 DecisionTreeClassifier에 전달하여 분류 학습
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['pclass', 'fare', 'age']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['embarked'])])),
                ('model', DecisionTreeClassifier(random_state=42))])

# 데이터 학습

In [15]:
# 4. 학습
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['pclass', 'fare', 'age']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['embarked'])])),
                ('model', DecisionTreeClassifier(random_state=42))])

# 예측값 산출 및 평가지표 설정

In [16]:
# 5. 평가 (테스트 데이터 기준)

y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('--- 테스트 데이터 평가 결과 ---')
print(f'Accuracy  (정확도)        : {accuracy:.4f}')
print(f'Precision (정밀도)        : {precision:.4f}')
print(f'Recall    (재현율)        : {recall:.4f}')
print(f'F1 score  (F1 점수)       : {f1:.4f}')


--- 테스트 데이터 평가 결과 ---
Accuracy  (정확도)        : 0.6760
Precision (정밀도)        : 0.6429
Recall    (재현율)        : 0.4865
F1 score  (F1 점수)       : 0.5538


# 사용자 입력 테스트

In [17]:
# ============================================================
# 6. 사용자 입력 예측
# ============================================================
# 아래 셀을 실행하면 pclass, fare, age, embarked 를 입력받아 survived 를 예측합니다.
# (Jupyter에서 이 셀만 단독 실행하세요)

pclass = float(input('객실 등급(pclass, 1~3)을 입력하세요: '))
fare = float(input('요금(fare)을 입력하세요: '))
age = float(input('나이(age)를 입력하세요: '))
embarked = input('탑승 항구(embarked: C / Q / S)를 입력하세요: ')

# 입력값을 DataFrame 형태로 만들어 pipeline 에 전달
user_input = pd.DataFrame(
    [[pclass, fare, age, embarked]],
    columns=['pclass', 'fare', 'age', 'embarked']
)
predicted = int(pipeline.predict(user_input)[0])

label = '생존' if predicted == 1 else '사망'
print(f'\n예측 결과(survived): {predicted} ({label})')


객실 등급(pclass, 1~3)을 입력하세요: 2
요금(fare)을 입력하세요: 10
나이(age)를 입력하세요: 20
탑승 항구(embarked: C / Q / S)를 입력하세요: Q

예측 결과(survived): 1 (생존)
